# LLM Text Annotation: Judging and Improving Output Quality with Local Fine-Tuning

Labelling text by hand is slow, so researchers ask models to do it. This notebook takes one labelling task, measures how well a small open model does it, fine-tunes the model to do it better, and then checks whether any of that survives a move to a different set of comments. Every model run was done once on a graphics card and saved, so the notebook reads the saved answers and gives the same numbers every time.

In [1]:
import json, sys
from pathlib import Path

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from model_call_scripts.cached_run import (
    RUBRIC, MODEL_ID, ADAPTER, ADAPTER_BALANCED, SEED, CACHE_DIR,
    N_EVAL_PER_CLASS, N_PROBE_PER_CLASS, annotate, cached_records, gold_splits,
)

DATA = Path("data")
ARTIFACTS = DATA / "artifacts"
MEDIA = Path("media")
TRAIN_STEPS = 200

# Where each model's bar for saying "yes" was moved to, measured once on a separate set of
# 100 New York Times pairs that is scored nowhere else in this notebook. Full precision on
# purpose: rounding these moves comments across the bar and changes the scores.
NYT_BAR = {"zero-shot": 0.964469850063324,
           "fine-tuned": 0.9951679110527039,
           "length-balanced": 0.7826507091522217}

pd.set_option("display.max_colwidth", 90)

for k, v in {
    "model": MODEL_ID,
    "seed": SEED,
    "eval set": f"{2 * N_EVAL_PER_CLASS} comments ({N_EVAL_PER_CLASS} per class)",
    "probe set": f"{2 * N_PROBE_PER_CLASS} comments",
    "saved answers": f"{len(list(CACHE_DIR.glob('*.json')))} model runs in cache/",
}.items():
    print(f"{k:>14}: {v}")

         model: unsloth/Qwen3-1.7B-unsloth-bnb-4bit
          seed: 20260810
      eval set: 400 comments (200 per class)
     probe set: 300 comments
 saved answers: 3 model runs in cache/


## 1. Labelling comments at scale

The SFU Opinion and Comments Corpus collects reader comments from *The Globe and Mail*. Suppose we want to know whether comment sections get less constructive during election campaigns. Before we can answer that, every comment needs a label, and there are far too many to read.

In [2]:
articles = pd.read_csv(DATA / "globe_articles.csv",
                       usecols=["article_id", "title", "published_date", "ncomments"])
gold = pd.read_csv(DATA / "socc_gold_labels.csv")

print(f"articles in the corpus : {len(articles):,}")
print(f"comments on them       : {articles.ncomments.sum():,.0f}")
print(f"hand-labelled by humans: {len(gold):,}")

articles in the corpus : 10,339
comments on them       : 663,173
hand-labelled by humans: 1,043


Someone labelled 1,043 of them by hand. That set is our answer key, and the rest of the notebook is about whether a model can extend it to the other 662,000.

A comment counts as **constructive** if it tries to add something: a specific point, evidence, a personal experience, a proposed solution. A comment is **not constructive** if it is only an insult, a one-line dismissal, or sarcasm with nothing behind it. Here is one of each.

In [3]:
for label in ("yes", "no"):
    row = gold[gold.is_constructive == label].iloc[1]
    print(f"[constructive: {label}]  {row.comment_text[:290]}\n")

[constructive: yes]  Everyone is still missing the point of what the Apple Watch is:It's fashion. It is by definition of no utility. Watches have been more fashion than function for as long as they've been worn. Anyone remember paying $50 for a Swatch that cost $2 to make? The Apple Watch actually offers quite

[constructive: no]  You may be using a blackberry. I'm still using a gooseberry.



In [4]:
print(gold.is_constructive.value_counts().to_string())
print(f"\ndrawn from {gold.article_id.nunique()} articles")

eval_set, probe_set = gold_splits(gold)
print(f"\nevaluation set: {len(eval_set)} comments, half of them constructive")
print(f"probe set     : {len(probe_set)} comments, kept separate")

is_constructive
yes    554
no     489

drawn from 13 articles

evaluation set: 400 comments, half of them constructive
probe set     : 300 comments, kept separate


The already labelled articles (which act as our answer key) are nearly balanced, and it comes from only a handful of articles, so it is a narrow slice of the corpus. We split it once, now. The evaluation set is scored once per method and never decides anything: no setting and no model is chosen by looking at it. The probe set is used later to watch the model during training.

## 2. Asking a model to annotate

The model is **Qwen3-1.7B**, an open-weights model small enough to run on a home graphics card. It is stored at **4-bit precision**, which squeezes each weight into a quarter of the usual space and shrinks the model to about 1.3 GB. Because of this each weight is recorded more coarsely, so the model's answers move around a little, and in return the whole thing fits in memory we actually have.

**Zero-shot** means we describe the task and ask, with no examples and no training. It is the cheapest of three ways to point a model at a task. **Few-shot** puts a handful of labelled examples in the prompt itself so the model can copy the pattern. **Fine-tuning** changes the model's weights, which is what section 4 does. We start zero-shot because it costs nothing to try, and because it tells us what the model already does before we change anything.

In [5]:
print(RUBRIC)

You are annotating reader comments from a Canadian news website.

A comment is CONSTRUCTIVE if it tries to add something to the conversation: it makes a specific point, gives evidence or a personal experience, offers a solution, or engages with the article's argument.

A comment is NOT CONSTRUCTIVE if it is only an insult, a one-line dismissal, sarcasm with no substance, off-topic ranting, or an unsupported assertion.

Comment:
"""{comment}"""

Reply in exactly this format and nothing else:
LABEL: yes
REASON: <one short sentence>


Each comment went through that prompt once, and the answer was saved under a key made from the model, the prompt and the comment. `annotate` reads those answers back.

In [6]:
zero_shot = annotate(eval_set.comment_counter, RUBRIC)

pd.DataFrame({"comment": eval_set.comment_text.str[:70],
              "human": eval_set.is_constructive,
              "model": zero_shot}).head(8)

,comment,human,model
0,Plenty. Ever been to Vancouver? There are condo boards that refuse to,no,no
1,Great piece! Thanks.,no,no
2,Tail wagging the dog,no,no
3,"Apparently, trying not to offend and be politically correct just doesn",no,no
4,The Belgian jihadis come mainly from the Rif mountains in northern Mor,no,no
5,Just keep whining - next thing that will happen is a ban on foreign bu,no,no
6,Why does the Globe and Mail even publish such a simplistic and accusat,no,no
7,"ROTFLMAO - hellloooo , no insurance, tax fraud,,, great starts, not to",no,no


It is worth looking at what the model actually wrote, not just the label we parsed out of it. Asking for a fixed format is what makes the output usable as data.

In [7]:
records = cached_records(RUBRIC)
for cid in eval_set.comment_counter.head(3):
    print(records[cid]["raw"], "\n" + "-" * 60)

LABEL: no
REASON: The comment is not constructive because it is an insult and lacks specific evidence or a personal experience. 
------------------------------------------------------------
LABEL: no
REASON: The comment is a one-line dismissal and does not add anything to the conversation. 
------------------------------------------------------------
LABEL: no
REASON: The comment is a sarcastic and unsupported assertion. 
------------------------------------------------------------


## 3. Judging the labels

Accuracy is the share the model got right. If the humans call half the comments constructive and the model says "not constructive" most of the time, the two will still land on the same answer fairly often without the model knowing anything.

**Cohen's kappa** subtracts that free agreement:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

$p_o$ is the **observed agreement**, the share of comments the two raters labelled identically. $p_e$ is the **expected agreement**, the share they would have landed on together if each had labelled independently at their own rate of saying yes. So $p_o - p_e$ is the agreement the model earned, and $1 - p_e$ is the agreement that was available to earn. Kappa is the ratio of the two: 0 means no better than guessing at that rate, and 1 means perfect.

The third number below is not a score. **The share the model calls constructive** just records how often it says yes, and on its own it cannot be right or wrong. Read it against the rate the humans use, which is one in two here. A model that answers the question should land near that. A model that has fallen back on one answer will not, and that gap is exactly what pulls kappa away from accuracy.

In [8]:
from sklearn.metrics import cohen_kappa_score


def as_binary(labels):
    """"yes"/"no", or something already 0/1, as a 0/1 array."""
    a = np.asarray(list(labels))
    if a.dtype.kind in "iub":
        return a.astype(int)
    return (np.char.lower(a.astype(str)) == "yes").astype(int)


def kappa(human, model):
    """Cohen's kappa, vectorised over a leading draw axis so resampling is cheap later."""
    po = (human == model).mean(axis=-1)
    ph, pm = human.mean(axis=-1), model.mean(axis=-1)
    pe = ph * pm + (1 - ph) * (1 - pm)
    return np.where(pe < 1, (po - pe) / np.where(pe < 1, 1 - pe, 1), 0.0)


def scoreboard(human, models):
    """The three numbers this notebook argues from, one column per model."""
    h = as_binary(human)
    return pd.DataFrame({
        name: {"accuracy": (h == as_binary(m)).mean(),
               "agreement (kappa)": float(kappa(h, as_binary(m))),
               "calls it constructive": as_binary(m).mean()}
        for name, m in models.items()})


# our kappa is the textbook one, so it should match the library's to floating point
assert abs(float(kappa(as_binary(eval_set.is_constructive), as_binary(zero_shot)))
           - cohen_kappa_score(eval_set.is_constructive, zero_shot)) < 1e-12

scoreboard(eval_set.is_constructive, {"zero-shot": zero_shot}).round(3)

,zero-shot
accuracy,0.73
agreement (kappa),0.46
calls it constructive,0.27


That 0.46 came from 400 comments. A different 400 would have given a different number, so the number on its own is not the whole answer. Resampling those 400 comments with replacement ten thousand times, and recomputing kappa each time, says how much it would move. We show a 95% confidence interval:

In [9]:
def fmt_ci(lo, hi):
    return f"[{lo:.2f}, {hi:.2f}]"


def bootstrap_kappa_ci(human, model, n_boot=10_000, alpha=0.05):
    """Kappa, plus the middle 95% of the kappas we get from resampling the comments."""
    h, m = as_binary(human), as_binary(model)
    draws = np.random.default_rng(SEED).integers(0, len(h), (n_boot, len(h)))
    lo, hi = np.percentile(kappa(h[draws], m[draws]),
                           [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"kappa": float(kappa(h, m)), "lo": lo, "hi": hi, "n": len(h)}


zs_ci = bootstrap_kappa_ci(eval_set.is_constructive, zero_shot)

print(f"kappa {zs_ci['kappa']:.3f}, 95% CI {fmt_ci(zs_ci['lo'], zs_ci['hi'])}"
      f"  (from {zs_ci['n']} comments)")

kappa 0.460, 95% CI [0.38, 0.54]  (from 400 comments)


Kappa around 0.46 counts as poor agreement on the scale from Landis and Koch (1977), who called 0.41 to 0.60 "moderate", 0.61 to 0.80 "substantial", and anything under 0.20 "slight". Important to keep in mind that these bands are suggestions, not ground truth. Nothing in the formula makes 0.60 a threshold, and a kappa that is fine for sorting comments into rough piles may be nowhere near good enough to publish a claim on. Treat them as vocabulary for talking about a number, and judge the number against what you plan to do with it.

The confusion matrix shows what shape the errors take.

In [10]:
counts = pd.crosstab(pd.Series(list(eval_set.is_constructive)), pd.Series(list(zero_shot)))
counts = counts.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
counts.index = ["humans said constructive", "humans said not constructive"]
counts.columns = ["model said constructive", "model said not constructive"]
counts.index.name = counts.columns.name = None
counts["share the model agreed"] = (np.diag(counts.to_numpy()) / counts.sum(axis=1)).round(2)
counts

,model said constructive,model said not constructive,share the model agreed
humans said constructive,100,100,0.50
humans said not constructive,8,192,0.96


Rows are the humans and columns are the model, so the top right cell is 100 comments the humans called constructive and the model called not constructive. The last column is each row's own agreement rate.



Look at the last column. The model agrees with the humans on 96% of the comments they called not constructive, and on half of the ones they called constructive. It is not randomly unreliable. It has one habit: saying "not constructive" too often. That habit causes almost all of its errors.

Before deciding whether 0.46 is bad, we need something to compare it against. About a fifth of the answer key was labelled a second time by an expert, and the crowd and the expert do not always agree either.

In [11]:
from sklearn.metrics import cohen_kappa_score

has_expert = gold[gold.expert_is_constructive.notna()]
crowd = (has_expert.is_constructive.str.lower() == "yes").astype(int)
expert = (has_expert.expert_is_constructive.str.lower() == "yes").astype(int)

print(f"{len(has_expert)} comments were labelled twice")
print(f"crowd and expert agree on {(crowd == expert).mean():.1%} of them")
print(f"as kappa, that is {cohen_kappa_score(crowd, expert):.3f}")

214 comments were labelled twice
crowd and expert agree on 78.5% of them
as kappa, that is 0.583


In [12]:
expert_ci = bootstrap_kappa_ci(crowd, expert)

print(f"crowd vs expert: kappa {expert_ci['kappa']:.3f}, "
      f"95% CI {fmt_ci(expert_ci['lo'], expert_ci['hi'])}  "
      f"(from only {expert_ci['n']} comments)")

crowd vs expert: kappa 0.583, 95% CI [0.48, 0.68]  (from only 214 comments)


Two humans working from the same instructions agree at 0.58, and even that is measured on 214 comments, so it carries an interval from 0.48 to 0.68. That is the bar the model is really being held to. Perfect agreement was never available, because the humans do not have it either.

## 4. Fine-tuning with QLoRA

Prompting can only rearrange what the model already does. Fine-tuning changes the model itself, by showing it labelled examples and adjusting its weights when it gets them wrong.

Doing that the ordinary way means updating all 1.7 billion weights, which needs far more memory than we have. **QLoRA** avoids it: the original weights stay frozen at 4 bits, and we attach small trainable matrices onto the frozen model and train only those. Under 2% of the model is trainable, and what we save at the end is one small file of adapter weights.

The training comments come from C3, a larger set of 12,000 comments labelled the same way.

In [13]:
LORA = dict(
    r=16,                      # size of the bolted-on matrices
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",     # attention
                    "gate_proj", "up_proj", "down_proj"],       # and the MLP
)

TRAIN_ARGS = dict(
    per_device_train_batch_size=2,     # 8 GB of VRAM does not allow more
    gradient_accumulation_steps=8,     # so accumulate to an effective batch of 16
    max_steps=TRAIN_STEPS,
    learning_rate=2e-4,
    fp16=True, bf16=False,             # Turing GPUs have no native bf16
    gradient_checkpointing=True,       # trade compute for memory
)

The **adapter** is the result: one small file of weights, a few tens of megabytes, that sits on top of the frozen model. Swapping adapters swaps annotators without touching the 1.3 GB underneath, which is how section 8 can compare two fine-tuned models easily.

Each training example is the same prompt we have been using, paired with the human label. The loss is computed **only on the label**, so the model is never rewarded for reproducing the prompt back to us.

In [14]:
train_log = json.loads((ARTIFACTS / "train_log.json").read_text())

print(train_log["trainable"])
print(f"{train_log['n_train']} training examples, {TRAIN_STEPS} steps")
print(f"{train_log['train_seconds'] / 60:.1f} minutes at {train_log['seconds_per_step']} s/step")
print(f"peak VRAM {train_log['peak_vram_mib']} MiB, final loss {train_log['final_loss']}")

17,432,576 trainable of 1,052,238,848 (1.66%)
2000 training examples, 200 steps
23.9 minutes at 7.18 s/step
peak VRAM 4899 MiB, final loss 0.2068


## 5. When did training stop helping?

A falling loss tells us the model fitted the training data. It does not tell us whether the labels got better. So during training we paused every 20 steps and pushed the same 300 probe comments through the model, recording the answer it would have given. Those comments were never trained on.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# One colour per class, used by every figure from here on. Blue against orange is the
# safest pair for the common forms of colour blindness.
YES, NO = "#2a78d6", "#eb6834"                  # constructive, not constructive
SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
MUTED, GRID, AXIS = "#898781", "#e1e0d9", "#c3c2b7"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK_2,
    "axes.titlecolor": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "grid.linestyle": "-", "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.frameon": False, "legend.fontsize": 9,
    "font.family": "sans-serif", "figure.dpi": 110,
})

checkpoints = dict(np.load(ARTIFACTS / "probe_checkpoints.npz"))
steps, p_yes, probe_labels = (checkpoints["steps"], checkpoints["p_yes"],
                              checkpoints["labels"])

# How much of the label a straight line can recover from the model's state. Strongly
# regularised because there are 300 comments and 2,048 numbers each.
probe_accuracy = np.array([
    cross_val_score(LogisticRegression(max_iter=3000, C=0.01),
                    X.astype(np.float32), probe_labels, cv=5).mean()
    for X in checkpoints["hidden"]])

preds = (p_yes > 0.5).astype(int)
agreement = np.array([float(kappa(probe_labels, q)) for q in preds])

# the shaded band: resample the 300 probe comments 4,000 times, once, and reuse those same
# draws at every checkpoint so the curve is compared against itself
draws = np.random.default_rng(SEED).integers(0, len(probe_labels),
                                             (4000, len(probe_labels)))
band = np.array([np.percentile(kappa(probe_labels[draws], q[draws]), [2.5, 97.5])
                 for q in preds])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2),
                               gridspec_kw={"width_ratios": [1.35, 1]})
ax1.fill_between(steps, band[:, 0], band[:, 1], color=YES, alpha=0.16, linewidth=0)
ax1.plot(steps, agreement, color=YES, linewidth=2.4, marker="o", markersize=5,
         markeredgecolor=SURFACE, markeredgewidth=0.8)
ax1.set_ylim(0.25, 0.9)
ax1.set_xlabel("training step")
ax1.set_ylabel("agreement with humans (kappa)")
ax1.set_title("Does more training help?", loc="left")

ax2.plot(steps, probe_accuracy, color=YES, linewidth=2.4, marker="o", markersize=5,
         markeredgecolor=SURFACE, markeredgewidth=0.8)
ax2.set_ylim(0.4, 1.0)
ax2.set_xlabel("training step")
ax2.set_ylabel("probe accuracy")
ax2.set_title("What the model already knew", loc="left")

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(MEDIA / "learning.png", dpi=140, bbox_inches="tight")
plt.close(fig)

pd.DataFrame({"step": steps, "agreement (kappa)": agreement.round(3),
              "probe accuracy": probe_accuracy.round(3)})

,step,agreement (kappa),probe accuracy
0,0,0.373,0.837
1,20,0.620,0.850
2,40,0.520,0.850
3,60,0.667,0.857
4,80,0.673,0.860
5,100,0.660,0.863
6,120,0.667,0.863
7,140,0.680,0.863
8,160,0.687,0.860
9,180,0.693,0.863


![Left: agreement with the human labels on 300 held-out comments at each training checkpoint, with a shaded 95% confidence band that is widest at the start and narrows as agreement climbs. Right: how accurately a straight line can recover the human label from the model's internal state, which stays flat near 0.85 across every checkpoint.](media/learning.png)

Almost all of the gain arrives in the first 60 steps. Agreement climbs from 0.37 to 0.67 and then stops. We could have trained for a third as long and finished with a similar annotator.

The shaded band is a 95% confidence interval, built by resampling the 300 probe comments with replacement 4,000 times and recomputing kappa each time. It is wide, because 300 comments is not many. Read the curve as a rough shape rather than as a step-by-step record: neighbouring checkpoints sit well inside each other's bands, so the small rise and fall between steps 20 and 60 is not something to explain.

The right panel asks something else. If we take the model's internal state and try to recover the human label from it with a straight line, how well does that work? This is called a **linear probe**, and it is a question about the model's representation rather than about its answers. Barely better at the end than at the start, 0.84 to 0.86. The two groups were already separable inside the model before we trained anything. Training changed the model's answers. It did not change what the model could tell apart.

## 6. Does it label better?

The same 400 comments, the same prompt, but this time using the model we just trained.

In [16]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

fine_tuned = annotate(eval_set.comment_counter, RUBRIC, adapter=ADAPTER)
truth = eval_set.is_constructive

# One square per comment, the wrong ones picked out. Comments are ordered so the ones
# humans called constructive sit above the line, which makes a lopsided error pattern
# visible as a block rather than as a number.
order = np.argsort(np.asarray(truth) != "yes", kind="stable")
in_order = np.asarray(truth)[order]
n = len(in_order)
cols = int(np.ceil(np.sqrt(n)))
rows = int(np.ceil(n / cols))
split = (in_order == "yes").sum() / cols - 0.5
right = "#dbe7f6"

fig, axes = plt.subplots(1, 2, figsize=(10.2, 5.0))
counts = []
for ax, (name, pred) in zip(axes, (("zero-shot", zero_shot), ("fine-tuned", fine_tuned))):
    wrong = (np.asarray(pred)[order] != in_order).astype(int)
    counts.append(int(wrong.sum()))
    padded = np.full(rows * cols, np.nan)
    padded[:n] = wrong
    ax.imshow(padded.reshape(rows, cols), cmap=ListedColormap([right, NO]),
              vmin=0, vmax=1, interpolation="nearest")
    ax.axhline(split, color=INK, linewidth=1.2)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_title(f"{name}\n{wrong.sum()} of {n} comments wrong", loc="left")

axes[0].text(-0.8, split / 2, "humans said\nconstructive", ha="right", va="center",
             fontsize=9, color=INK_2)
axes[0].text(-0.8, (split + rows) / 2, "humans said\nnot constructive", ha="right",
             va="center", fontsize=9, color=INK_2)
fig.legend(handles=[Patch(facecolor=right, label="model agreed with the humans"),
                    Patch(facecolor=NO, label="model got it wrong")],
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Each square is one comment", x=0.125, ha="left", fontsize=12, color=INK)
fig.tight_layout(rect=[0, 0.04, 1, 0.95])
fig.savefig(MEDIA / "before_after.png", dpi=140, bbox_inches="tight")
plt.close(fig)

wrong_before, wrong_after = counts
fixed = sum(z != t and f == t for t, z, f in zip(truth, zero_shot, fine_tuned))
broke = sum(z == t and f != t for t, z, f in zip(truth, zero_shot, fine_tuned))

print(f"zero-shot got {wrong_before} of {len(truth)} wrong; fine-tuned got {wrong_after} wrong")
print(f"fine-tuning fixed {fixed} comments and broke {broke}")

zero-shot got 108 of 400 wrong; fine-tuned got 47 wrong
fine-tuning fixed 72 comments and broke 11


![Two grids of 400 squares, one per evaluation comment, with the squares the model got wrong picked out. Comments humans called constructive sit above the dividing line. The zero-shot grid is heavily marked across its top half; the fine-tuned grid is mostly clear.](media/before_after.png)

Each square is one comment, and the ones above the line are those humans called constructive. After fine-tuning the top half mostly clears which was our orignal big issue.

The grid says how many errors there were before and after. It cannot say whether they are the same errors. 47 errors after could be 47 of the original 108 surviving, or it could be 36 surviving and 11 brand new ones. Those are different stories about what fine-tuning did, and this figure separates them.

![Three columns of the 400 evaluation comments: the human label, then the zero-shot label, then the fine-tuned label. Ribbons are coloured by the human label. A wide blue ribbon leaves the constructive column and arrives at the zero-shot not-constructive column, then most of it returns to constructive after fine-tuning. The two paths that changed the model's mind are outlined.](media/sankey.png)


Follow the blue. 200 comments were called constructive by the humans, and 108 of them, came out of the zero-shot column labelled not constructive. That single ribbon is most of the zero-shot error, where it labelled as non-constructive too often. After fine-tuning most of that ribbon crosses back. The orange going the other way is much thinner: fine-tuning broke 11 comments while fixing 72.

The two outlined paths are the interesting ones to read.

In [17]:
def flow_table(flows, before, after, chars=200):
    """The comments on one path, for reading rather than counting.

    before="wrong", after="right" is the set fine-tuning fixed; the reverse is what it broke.
    """
    rows = flows[(flows.zs_ok == before) & (flows.ft_ok == after)]
    said = lambda col: np.where(rows[col] == "yes", "constructive", "not constructive")
    shorten = lambda t: " ".join(str(t).split())
    return pd.DataFrame({
        "humans said": said("truth"),
        "zero-shot said": said("zero_shot"),
        "fine-tuned said": said("fine_tuned"),
        "comment": [s if len(s) <= chars else s[:chars - 1] + "\u2026"
                    for s in map(shorten, rows.text)],
    }).reset_index(drop=True)


# one row per comment, with its label under each of the three raters
flows = pd.DataFrame({"truth": np.asarray(truth),
                      "zero_shot": np.asarray(zero_shot),
                      "fine_tuned": np.asarray(fine_tuned),
                      "text": np.asarray(eval_set.comment_text)})
flows["zs_ok"] = np.where(flows.zero_shot == flows.truth, "right", "wrong")
flows["ft_ok"] = np.where(flows.fine_tuned == flows.truth, "right", "wrong")

print(f"{fixed} comments fine-tuning fixed. The first few:")
flow_table(flows, "wrong", "right").head(4)

72 comments fine-tuning fixed. The first few:


,humans said,zero-shot said,fine-tuned said,comment
0,constructive,not constructive,constructive,Simpson is having his little joke by encouraging the NDP to play with this expensive h...
1,constructive,not constructive,constructive,"Paul The first world relates to per capita income, and now how that income was achieve..."
2,constructive,not constructive,constructive,"Meanwhile, on the migrant file and Euro-terrorism file in the past 24 hours or so: Isl..."
3,constructive,not constructive,constructive,It will be a curious time in 4 years from now: voters in the Midwest who elected Trump...


Eleven comments is few enough to read all of them, which is worth doing. A model that gets better on average still gets specific things wrong that it used to get right, and the only way to know whether that trade is acceptable is to look at what was traded.

In [18]:
print(f"all {broke} comments fine-tuning broke:")
flow_table(flows, "right", "wrong")

all 11 comments fine-tuning broke:


,humans said,zero-shot said,fine-tuned said,comment
0,not constructive,not constructive,constructive,Please read this article and tell me if there is a double standard applied to China in...
1,constructive,constructive,not constructive,Perhaps Trump's supporters here might check their triumphalism at the knowledge that m...
2,not constructive,not constructive,constructive,"Of course we all know, unfortunately, there is a segment of the voting public that wor..."
3,not constructive,not constructive,constructive,The 15% tax helps pay for things like Professor Yu's money from the Canadian Federal G...
4,not constructive,not constructive,constructive,Tell you daughters that it was a free and fair election and that 42% of all women vote...
5,not constructive,not constructive,constructive,Your article “As technology marches”. . . mirrors my own emotional / psychological exp...
6,not constructive,not constructive,constructive,"'He’s hardly the choice of the establishment, media, cultural, business or political'a..."
7,constructive,constructive,not constructive,This article reinforces the point that the most significant deficit in this country at...
8,not constructive,not constructive,constructive,So we are going to compare an unavoidable head tax on a group of poor and disadvantage...
9,not constructive,not constructive,constructive,'...whether the federal budget shows a small deficit or a small surplus is almost tota...


Let's now look at the fine-tuned model's accuracy, kappa and how often it calls a comment constuctive and then the confidence intervals for each.

In [19]:
scoreboard(truth, {"zero-shot": zero_shot,
                   "fine-tuned": fine_tuned}).round(3)

,zero-shot,fine-tuned
accuracy,0.73,0.882
agreement (kappa),0.46,0.765
calls it constructive,0.27,0.438


In [20]:
def bootstrap_kappa_diff_ci(human, a, b, n_boot=10_000, alpha=0.05):
    """Interval on kappa(b) - kappa(a) when both were scored on the same comments.

    Each replicate draws one set of comments and scores both raters on it, so a resample
    that happens to be easy is easy for both and the shared difficulty cancels. Treating
    them as two independent samples would widen this a lot and would be the wrong test.
    """
    h, a, b = as_binary(human), as_binary(a), as_binary(b)
    draws = np.random.default_rng(SEED).integers(0, len(h), (n_boot, len(h)))
    hd = h[draws]
    lo, hi = np.percentile(kappa(hd, b[draws]) - kappa(hd, a[draws]),
                           [100 * alpha / 2, 100 * (1 - alpha / 2)])
    ka, kb = float(kappa(h, a)), float(kappa(h, b))
    return {"a": ka, "b": kb, "diff": kb - ka, "lo": lo, "hi": hi, "n": len(h)}


ft_ci = bootstrap_kappa_ci(truth, fine_tuned)
gain = bootstrap_kappa_diff_ci(truth, zero_shot, fine_tuned)

print(f"zero-shot  kappa {zs_ci['kappa']:.3f}  {fmt_ci(zs_ci['lo'], zs_ci['hi'])}")
print(f"fine-tuned kappa {ft_ci['kappa']:.3f}  {fmt_ci(ft_ci['lo'], ft_ci['hi'])}")
print(f"gain            {gain['diff']:+.3f}  {fmt_ci(gain['lo'], gain['hi'])}"
      "   (paired: both models scored on each resample of the same comments)")

zero-shot  kappa 0.460  [0.38, 0.54]
fine-tuned kappa 0.765  [0.70, 0.82]
gain            +0.305  [0.23, 0.38]   (paired: both models scored on each resample of the same comments)


Fine-tuning works. Kappa goes from 0.46 to 0.77, the two intervals do not overlap, and the model now calls 44% of comments constructive against a true rate of 50%, so the habit is gone.

The crowd and the expert agreed with each other at kappa 0.58, which is lower than the model's 0.77. It is tempting to read that as the model agreeing with the crowd more closely than the expert does. Before believing it, notice that those two numbers were not measured on the same comments: 0.77 comes from the 400 evaluation comments, 0.58 from the 214 that were labelled twice. Comparing them directly is comparing two different exams. 81 comments are in both sets, so we can run the comparison properly.

In [21]:
# the comments that are in both the evaluation set and the doubly-labelled set
shared = eval_set[eval_set.comment_counter.isin(has_expert.comment_counter)]
twice = has_expert.set_index("comment_counter").loc[shared.comment_counter]

crowd_here = twice.is_constructive.str.lower()
expert_here = twice.expert_is_constructive.str.lower()
model_here = annotate(shared.comment_counter, RUBRIC, adapter=ADAPTER)

versus = bootstrap_kappa_diff_ci(crowd_here, expert_here, model_here)
print(f"on the {versus['n']} comments in both sets, agreement with the crowd:")
print(f"  the expert    {versus['a']:.3f}")
print(f"  the model     {versus['b']:.3f}")
print(f"  difference    {versus['diff']:+.3f}  {fmt_ci(versus['lo'], versus['hi'])}")

on the 81 comments in both sets, agreement with the crowd:
  the expert    0.546
  the model     0.824
  difference    +0.278  [0.12, 0.44]


On the same 81 comments, scored against the same crowd labels, the model agrees at 0.82 and the expert at 0.55. The interval on the difference runs from +0.12 to +0.44 and does not include zero, so the claim survives a proper test. 

What the claim means has not changed, though. The model was trained on crowd labels, so agreeing with the crowd is what it was built to do. The expert disagrees with the crowd in places because the expert has a different and probably better reading of the rubric. Beating the expert at matching the crowd is a statement about who the model imitates, not about who is right.

## 7. Does it work on different data?

A label is only useful if it means the same thing on data the model has not seen. So we move to *The New York Times*, where editors mark a small number of reader comments as picks.

We turn that into a game. Take two comments from the same article, one an editor's pick and one not, and ask which is which. Guessing gets 50%. The two comments in a pair are also matched on length, within 20% of each other, so a model cannot win this game by simply preferring longer comments.

In [22]:
# The scores were produced once on a GPU and saved, keyed the same way the labels are.
COLUMNS = {"zero-shot": "zero-shot", "fine-tuned": "naive_adapter",
           "length-balanced": "balanced_adapter"}

pairs = pd.read_csv(DATA / "nyt_pairs.csv")
pairs["comment_id"] = pairs.comment_id.astype(str)
saved = np.load(ARTIFACTS / "nyt_p_yes.npz", allow_pickle=True)
row_of = {c: i for i, c in enumerate(saved["comment_id"].astype(str))}
order = [row_of[c] for c in pairs.comment_id]
scored = pairs.assign(**{name: saved[key][order] for name, key in COLUMNS.items()})

print(f"{len(scored) // 2:,} pairs from {scored.article_id.nunique()} articles, "
      f"{len(scored):,} comments\n")

# Two real pairs. In each one, which comment did an editor pick?
answers = []
for n, pair_id in enumerate(scored.pair_id.unique()[:2], start=1):
    both = scored[scored.pair_id == pair_id].sample(frac=1, random_state=SEED + n)
    answers.append("AB"[list(both.is_pick).index("yes")])
    print(f"--- pair {n} ---")
    for letter, (_, row) in zip("AB", both.iterrows()):
        print(f"  {letter}: {row.comment_text[:260]}\n")

print("editor picked:", ", ".join(f"pair {i} = {a}" for i, a in enumerate(answers, 1)))

1,001 pairs from 366 articles, 2,002 comments

--- pair 1 ---
  A: For decades, Trump viewed the law with contempt. It certainly never constrained him from stiffing contractors or committing fraud. And Trump knew how to use his money and his lawyers to bully and threaten anyone who stood in his way.
It always worked for him.


  B: Oh I see, so the FBI is suddenly biased in favor of Democrats? Is that why Comey (a registered Republican) delivered a public rebuke of Hillary during the campaign, breaking with FBI norms and guidelines, rather than simply state that they could find no basis 

--- pair 2 ---
  A: Nothing so represents the Trump administration better than Mick Mulvaney's performance this afternoon, fresh from Ash Wednesday service with a cross smudged on his forehead, telling congress today that Trump's military parade will cost between 10M-30M, meanwhi

  B: If he's got to have his parade, then let him have it. Let him pay for it with food boxes and let him watch it alone. 

Here is how the models did across all 1,001 pairs.

In [23]:
def side_by_side(model):
    """One row per pair: the pick's score and its partner's."""
    return scored.pivot(index="pair_id", columns="is_pick", values=model)


def game_score(model):
    """Share of pairs where the model scored the editor's pick above its partner."""
    w = side_by_side(model)
    return float(((w["yes"] > w["no"]).astype(float) + 0.5 * (w["yes"] == w["no"])).mean())


pd.DataFrame({"scores the editor's pick higher":
              {"always guessing": 0.5,
               "zero-shot": game_score("zero-shot"),
               "fine-tuned": game_score("fine-tuned")}}).round(3)

,scores the editor's pick higher
always guessing,0.500
zero-shot,0.551
fine-tuned,0.595



**Is a model better than guessing?** Both models clear the bar: zero-shot wins 554 of every 1,000 decided pairs and fine-tuned wins 596. 

Maybe more analysis here

## 8. Now ask it to label them

Winning the game only requires ranking one comment above another. Actual annotation requires a straight answer on each comment by itself. So we ask the fine-tuned model the same question it was trained on, one comment at a time.

In [24]:
def calls_it_a_pick(model, bar=0.5):
    """Turn scores into yes/no answers at a given bar."""
    return np.where(scored[model] > bar, "yes", "no")


rows = {}
for name in ("zero-shot", "fine-tuned"):
    rows[name] = {
        "calls it a pick": float((scored[name] > 0.5).mean()),
        "agreement (kappa)": float(kappa(as_binary(scored.is_pick),
                                         as_binary(calls_it_a_pick(name)))),
    }
pd.DataFrame(rows).round(3)

,zero-shot,fine-tuned
calls it a pick,0.576,0.850
agreement (kappa),0.076,0.008


The fine-tuned model calls 85% of the comments an editor's pick, and its agreement collapses to 0.01. It is useless as an annotator here, on exactly the comments where it just won the game. Two things went wrong:

**First, it leaned on a clue that does not carry over across data.** In the Globe and Mail set, constructive comments were simply longer.

In [25]:
# how long is a comment of each class? the clue the model can learn instead
by_class = []
for label, name in (("yes", "constructive"), ("no", "not constructive")):
    lengths = gold.loc[gold.is_constructive == label, "comment_text"].str.len()
    by_class.append({"humans said": name, "comments": len(lengths),
                     "median length": f"{lengths.median():.0f} characters",
                     "longest tenth": f"over {lengths.quantile(0.9):.0f} characters"})
pd.DataFrame(by_class).set_index("humans said")

,comments,median length,longest tenth
humans said,,,
constructive,554,433 characters,over 1026 characters
not constructive,489,111 characters,over 243 characters


A constructive comment in the answer key is four times the length of a non-constructive one. A model can score well on that corpus by partly learning "long means constructive." At the Times, the two comments in a pair were matched for length on purpose.

So we rebuilt the training sample, keeping the same number of comments in each length band for both classes, and trained a second model on it. Everything else about the recipe was identical.

**Second, its bar for saying yes was in the wrong place.** The model learned to say yes where about half of comments qualify. At the Times almost every comment clears that bar. Moving the bar needs no retraining, just a separate set of 100 pairs, kept aside for this and scored nowhere else.

In [26]:
socc = {"zero-shot": zero_shot, "fine-tuned": fine_tuned,
        "length-balanced": annotate(eval_set.comment_counter, RUBRIC,
                                    adapter=ADAPTER_BALANCED)}

pd.DataFrame({name: {
    "Globe and Mail: agreement (kappa)": float(kappa(as_binary(truth), as_binary(answers))),
    "New York Times: wins the game": game_score(name),
    "New York Times: agreement, bar moved": float(kappa(
        as_binary(scored.is_pick), as_binary(calls_it_a_pick(name, NYT_BAR[name])))),
} for name, answers in socc.items()}).round(3)

,zero-shot,fine-tuned,length-balanced
Globe and Mail: agreement (kappa),0.460,0.765,0.555
New York Times: wins the game,0.551,0.595,0.618
"New York Times: agreement, bar moved",0.075,0.055,0.161


Three numbers are shown for each model because no one of them is the whole story. **Agreement (kappa)** is the one we care about for annotation, and it is the strictest. **Wins the game** only asks the model to rank one comment above another, which is easier and which is why every model does better on it. **Calls it a pick** is the rate at which the model says yes, and it is a sanity: a model can reach respectable agreement while calling far too many comments picks, and you would want to know that before running it over a million comments. The length-balanced model wins on some of these and loses on others, which is the point.

Read the two New York Times rows across, then look back at the first row. The length-balanced model has the best point estimates at the Times, 62% on the game and 0.16 agreement once its bar is moved. It is also clearly worse on the Globe and Mail comments, 0.56 against 0.77.

The first model was a crammer: it memorised what answers looked like on the practice exam, including a shortcut the exam happened to allow (comment lenght), and it aced that exam. The second model was denied the shortcut, scored lower on the practice exam, and did better on a different slightly different exam (NYT).

Here is one pair where the difference shows.

In [27]:
missed_by, separated_by = "fine-tuned", "length-balanced"
a, b = side_by_side(missed_by), side_by_side(separated_by)
bar_a, bar_b = NYT_BAR[missed_by], NYT_BAR[separated_by]

# a real pair the first model calls a pick twice and the second tells apart
ok = a.index[(a["yes"] > bar_a) & (a["no"] > bar_a)
             & (b["yes"] > bar_b) & (b["no"] <= bar_b)]
text = scored.pivot(index="pair_id", columns="is_pick", values="comment_text")
shortest = text.loc[ok].map(len).sum(axis=1).sort_values().index
pair_id = shortest[int(np.random.default_rng(SEED).integers(min(5, len(shortest))))]

print("EDITOR'S PICK :", text.loc[pair_id, "yes"][:280], "\n")
print("NOT PICKED    :", text.loc[pair_id, "no"][:280], "\n")
print(f"fine-tuned      scored them {a.loc[pair_id, 'yes']:.2f} and "
      f"{a.loc[pair_id, 'no']:.2f}  -> calls both a pick")
print(f"length-balanced scored them {b.loc[pair_id, 'yes']:.2f} and "
      f"{b.loc[pair_id, 'no']:.2f}  -> tells them apart")

EDITOR'S PICK : If the president was truly unconcerned about the Mueller investigation, he’d simply put it behind him and not talk or tweet about it. Running the country is what he should be doing and not second-guessing the Special Prosecutor.

Mueller’s next move is anybody’s guess. He could b 

NOT PICKED    : How can there be no charges of collusion for Trump? His son according to Steve Bannons book did a treasonous act meeting with the Russians for bad info on Hillary . None was found. Then Trump in his own loose lips on public tv during a rally asked Russia for help in finding 30,00 

fine-tuned      scored them 1.00 and 1.00  -> calls both a pick
length-balanced scored them 0.84 and 0.63  -> tells them apart


Add conclusion once approved

### Data and licences

- **SOCC** and its constructiveness subset: Kolhatkar, V., H. Wu, L. Cavasso, E. Francis, K. Shukla and M. Taboada (2020). The SFU Opinion and Comments Corpus: A corpus for the analysis of online news comments. *Corpus Pragmatics* 4(2), 155-190. https://doi.org/10.1007/s41701-019-00065-w Licensed CC BY-NC-SA 4.0.
- **C3**: Kolhatkar, V., N. Thain, J. Sorensen, L. Dixon and M. Taboada (2020). *C3: The Constructive Comments Corpus.* Jigsaw and Simon Fraser University. DOI: 10.25314/ea49062a-5cf6-4403-9918-539e15fd7b52 Licensed CC BY-NC 4.0.
- **New York Times comments**: Kesarwani, A. *New York Times Comments.* Kaggle. https://www.kaggle.com/datasets/aashita/nyt-comments Licensed CC BY-NC-SA 4.0.
- **Model**: Qwen3-1.7B, Apache 2.0, 4-bit build by Unsloth.

### References

- Fang, Q., J. Garcia Bernardo and E-J. van Kesteren (2026). *A Methodological Guide on Using Large Language Models for Text Annotation in the Social Sciences and Humanities with Python and R.* https://arxiv.org/abs/2604.09638
- Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). *QLoRA: Efficient finetuning of quantized LLMs.* https://arxiv.org/abs/2305.14314
- Hu, E., et al. (2021). *LoRA: Low-rank adaptation of large language models.* https://arxiv.org/abs/2106.09685
- Kolhatkar, V. and M. Taboada (2017). Using New York Times Picks to identify constructive comments. *Proceedings of the Workshop Natural Language Processing Meets Journalism, EMNLP.* https://www.aclweb.org/anthology/W17-4218/
- Krippendorff, K. (2018). *Content Analysis: An Introduction to Its Methodology.* SAGE. On treating annotation as measurement.